# Bronze Layer: ERP Customer (Auto Loader)

**Pattern**: Auto Loader (Streaming Ingestion)  
**Trigger**: AvailableNow (Incremental Batch)

In [0]:
from pyspark.sql.functions import current_timestamp

source_path = "/Volumes/workspace/bronze/raw_sources/source_erp/CUST_AZ12.csv"
target_table = "workspace.bronze.erp_cust_az12_cdc"
checkpoint_path = "/Volumes/workspace/bronze/checkpoints/erp_cust"
schema_path = "/Volumes/workspace/bronze/checkpoints/erp_cust_schema"

query = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()